<a href="https://colab.research.google.com/github/AlenpeTecMilenio/MercadoFresco/blob/main/MerFresBot_Avance1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
assert sys.version_info >= (3, 10), 'Se requiere Python 3.10 o superior.'
%pip -q install nltk==3.9.1
print('Python:', sys.version.split()[0])

Python: 3.13.16


In [11]:
import re
import unicodedata
import nltk
from nltk.chat.util import Chat, reflections
from IPython.display import display, Markdown

BASE = [
('saludo', r'^(hola|buenos dias|buenas tardes|buenas noches|hey)\b.*', '¡Hola! Soy MerFresBot. Consulta sobre: altas, actualizacion de datos, ordenes de compra, soporte tecnico.'),
('alta', r'.*\b(inscripcion|registro|alta|invitacion)\b.*', 'Consulta a tu contacto de compras y solicita que te mande una invitacion de registro al portal de proveedores de Mercado Fresco.'),
('actualizar mis datos', r'.*\b(actualizar|cambios|cambiar|reemplazar)\b.*', 'Inicia sesion con el usuario y contraseña que usaste para tramitar tu alta en el portal. Si existe algún problema regresa al menu principal y solicita soporte tecnico o comunicate con tu contacto de compras.'),
('orden de compra', r'.*\b(compra|oc|pedido|pedidos)\b.*', 'Ingresa al portal de proveedores de MercadoFresaco e ingresa a la sección de ordenes de compra, especifica tu numero de proveedor y haz clic en ver pedidos. Descarga en pdf copias de las ordenes de compra.'),
('soporte tecnico', r'.*\b(soporte|asistencia|asesoria|desbloqueo)\b.*', 'Para problemas de acceso al portal, haz clic en el link restablecimiento de contraseña o comunicate con tu contacto de compras.'),
('ayuda', r'.*\b(ayuda|menu|opciones|que puedes hacer)\b.*', 'Temas disponibles: alta, actualizar mis datos, orden de compra, y soporte tecnico. Escribe salir para terminar.'),
('agradecimiento', r'.*\bgracias\b.*', '¡De nada! ¿Necesitas orientación sobre otro tema?'),
('despedida', r'^(salir|adios|hasta luego)[.!?]*$', '¡Hasta luego! Gracias por consultar MerFresBot.'),
('vacio', r'^$', 'Escribe una consulta para poder orientarte.'),
('manuales', r'.*\b(biblioteca|manual|guia|instructivo)\b.*', 'En la seccion de manuales podras encontrar soporte documental para ayuda sobre la navegación en el portal de MercadoFresco.' ),
('desconocida', r'.*', 'No identifiqué tu consulta. Intenta mencionar registro, alta,actualizar mis datos, orden de compra o soporte tecnico.')
]

chatbot = Chat([(patron, [respuesta]) for _, patron, respuesta in BASE], reflections)

def normalizar(texto):
  texto = unicodedata.normalize('NFD', texto.lower().strip())
  texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
  return re.sub(r'\s+', ' ', texto)

def responder(texto):
  limpio = normalizar(texto)
  intencion = next(nombre for nombre, patron, _ in BASE if re.match(patron, limpio, re.IGNORECASE))
  return {'intencion': intencion, 'respuesta': chatbot.respond(limpio)}

for nombre, patron, respuesta in BASE:
    print(f'{nombre}: {respuesta}')
print('Versión NLTK:', nltk.__version__)

saludo: ¡Hola! Soy MerFresBot. Consulta sobre: altas, actualizacion de datos, ordenes de compra, soporte tecnico.
alta: Consulta a tu contacto de compras y solicita que te mande una invitacion de registro al portal de proveedores de Mercado Fresco.
actualizar mis datos: Inicia sesion con el usuario y contraseña que usaste para tramitar tu alta en el portal. Si existe algún problema regresa al menu principal y solicita soporte tecnico o comunicate con tu contacto de compras.
orden de compra: Ingresa al portal de proveedores de MercadoFresaco e ingresa a la sección de ordenes de compra, especifica tu numero de proveedor y haz clic en ver pedidos. Descarga en pdf copias de las ordenes de compra.
soporte tecnico: Para problemas de acceso al portal, haz clic en el link restablecimiento de contraseña o comunicate con tu contacto de compras.
ayuda: Temas disponibles: alta, actualizar mis datos, orden de compra, y soporte tecnico. Escribe salir para terminar.
agradecimiento: ¡De nada! ¿Necesit

In [12]:
CONVERSACIONES = [
[('Hola', 'saludo'), ('¿Como puedo darme de alta?', 'alta'), ('Gracias', 'agradecimiento')],
[('Buenas tardes', 'saludo'), ('¿Donde veo mis pedidos?', 'orden de compra')],
[('Necesito actualizar mis datos', 'actualizar mis datos'), ('Adiós', 'despedida')],
[('Quiero una invitacion', 'alta'), ('Gracias', 'agradecimiento')],
[('¿Donde desbloqueo mi usuario', 'soporte tecnico'), ('salir', 'despedida')],
[('¿Qué puedes hacer?', 'ayuda'), ('¿Cómo me registro?', 'alta')],
[('HOLA', 'saludo'), ('BUSCO MANUAL DE APOYO', 'manuales')],
[(' ', 'vacio'), ('Necesito copia del pedido', 'orden de compra')],
[('¿Qué clima hará mañana?', 'desconocida'), ('menú', 'ayuda')],
[('Quiero saber como obtener una invitacion de registro', 'alta'), ('hasta luego', 'despedida')],
[('Quiero cambiar mi cuenta bancaria','actualizar mis datos'), ('Adios','despedida')]
]
registros = []
for numero, conversacion in enumerate(CONVERSACIONES, 1):
  print(f'\n=== Conversación {numero:02d} ===')
  for entrada, esperada in conversacion:
    salida = responder(entrada)
    correcto = salida['intencion'] == esperada
    registros.append({'conversacion': numero, 'entrada': entrada, 'esperada': esperada, **salida,
    'correcto': correcto})
    print('Usuario:', repr(entrada))
    print('MerFresBot:', salida['respuesta'])
    print('Intención:', salida['intencion'], '| Resultado:', 'OK' if correcto else 'REVISAR')
aciertos = sum(r['correcto'] for r in registros)
print(f'\nResultado: {aciertos}/{len(registros)} turnos coinciden con lo esperado.')
assert aciertos == len(registros), 'Hay casos que requieren revisión.'


=== Conversación 01 ===
Usuario: 'Hola'
MerFresBot: ¡Hola! Soy MerFresBot. Consulta sobre: altas, actualizacion de datos, ordenes de compra, soporte tecnico.
Intención: saludo | Resultado: OK
Usuario: '¿Como puedo darme de alta?'
MerFresBot: Consulta a tu contacto de compras y solicita que te mande una invitacion de registro al portal de proveedores de Mercado Fresco.
Intención: alta | Resultado: OK
Usuario: 'Gracias'
MerFresBot: ¡De nada! ¿Necesitas orientación sobre otro tema?
Intención: agradecimiento | Resultado: OK

=== Conversación 02 ===
Usuario: 'Buenas tardes'
MerFresBot: ¡Hola! Soy MerFresBot. Consulta sobre: altas, actualizacion de datos, ordenes de compra, soporte tecnico.
Intención: saludo | Resultado: OK
Usuario: '¿Donde veo mis pedidos?'
MerFresBot: Ingresa al portal de proveedores de MercadoFresaco e ingresa a la sección de ordenes de compra, especifica tu numero de proveedor y haz clic en ver pedidos. Descarga en pdf copias de las ordenes de compra.
Intención: orden d

In [16]:
ACTIVAR_CHAT = False
if ACTIVAR_CHAT:
  print('MerFresBot: ¡Hola! Escribe tu consulta o salir.')
  while True:
    entrada = input('Tú: ')
    salida = responder(entrada)
    print('MerFresBot:', salida['respuesta'])
    if salida['intencion'] == 'despedida':
      break
else:
  print('Activa ACTIVAR_CHAT para conversar manualmente.')

MerFresBot: ¡Hola! Escribe tu consulta o salir.
Tú: invitacion
MerFresBot: Consulta a tu contacto de compras y solicita que te mande una invitacion de registro al portal de proveedores de Mercado Fresco.
Tú: alta
MerFresBot: Consulta a tu contacto de compras y solicita que te mande una invitacion de registro al portal de proveedores de Mercado Fresco.
Tú: Registro al portal
MerFresBot: Consulta a tu contacto de compras y solicita que te mande una invitacion de registro al portal de proveedores de Mercado Fresco.
Tú: Como veo mi pedido?
MerFresBot: Ingresa al portal de proveedores de MercadoFresaco e ingresa a la sección de ordenes de compra, especifica tu numero de proveedor y haz clic en ver pedidos. Descarga en pdf copias de las ordenes de compra.
Tú: cambio de contraseña
MerFresBot: No identifiqué tu consulta. Intenta mencionar registro, alta,actualizar mis datos, orden de compra o soporte tecnico.
Tú: Desbloqueo de usuario
MerFresBot: Para problemas de acceso al portal, haz clic en

In [17]:
import csv
import json
import zipfile
from pathlib import Path
from datetime import datetime, timezone

carpeta = Path('evidencias_merfresbot')
carpeta.mkdir(exist_ok=True)
with (carpeta / 'pruebasmerfresbot.csv').open('w', newline='', encoding='utf-8-sig') as f:
  escritor = csv.DictWriter(f, fieldnames=list(registros[0]))
  escritor.writeheader()
  escritor.writerows(registros)
lineas = ['Ejecución UTC: ' + datetime.now(timezone.utc).isoformat(), f'Python:   {sys.version.split()[0]} | NLTK: {nltk.__version__}']
for r in registros:
  lineas.extend([f"Conversación {r['conversacion']} | Usuario: {r['entrada']!r}", 'MerFresBot: ' +   r['respuesta'], f"Intención: {r['intencion']} | Esperada: {r['esperada']} | Correcto:   {r['correcto']}", ''])
  (carpeta / 'conversaciones.txt').write_text('\n'.join(lineas), encoding='utf-8')
with zipfile.ZipFile('Evidencias_MerFresBot.zip', 'w', zipfile.ZIP_DEFLATED) as z:
  for archivo in carpeta.iterdir():
    z.write(archivo, arcname=archivo.name)
print('Evidencias generadas: Evidencias_MerFresBot.zip')
DESCARGAR = False # Cambia a True si deseas descargar el ZIP en Colab.

if DESCARGAR:
  from google.colab import files
  files.download('Evidencias_MerFresBot.zip')

Evidencias generadas: Evidencias_MerFresBot.zip
